# Phase 4 — Microservice API
## Chiron-AI Hotel Assistant — FastAPI + WebSocket Backend

This notebook launches and tests the **containerized microservice** built in Phase 4.

| Deliverable | File |
|---|---|
| FastAPI service | `api/main.py` |
| WebSocket endpoint | `ws://localhost:8000/ws/chat` |
| LLM handler (async streaming) | `api/llm_handler.py` |
| Session + intent management | `api/session_store.py` |
| Pydantic message schemas | `api/models.py` |
| Dockerfile (pre-built wheel) | `Dockerfile` |
| Docker Compose config | `docker-compose.yml` |
| Postman test collection | `postman_collection.json` |

### Run order
1. **Cell 1** — install dependencies
2. **Cell 2** — project structure overview
3. **Cell 3** — start the server in a background thread
4. **Cells 4–8** — REST and WebSocket tests
5. **Cell 9** — concurrent users test
6. **Cell 10** — Docker instructions

---
## Cell 1 — Install Dependencies

In [ ]:
# Install Phase 4 dependencies.
# llama-cpp-python uses --prefer-binary to avoid C++ compilation.
# Restart the kernel after this cell if it's your first install.

%pip install fastapi "uvicorn[standard]" websockets httpx pydantic huggingface-hub --quiet

%pip install llama-cpp-python \
    --prefer-binary \
    --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu \
    --quiet

print("Done! Restart the kernel if this is your first install.")

---
## Cell 2 — Project Structure

```
Phase 4/
├── api/
│   ├── __init__.py           package marker
│   ├── main.py               FastAPI app, REST + WebSocket endpoints
│   ├── models.py             Pydantic schemas (ChatRequest, TokenChunk, etc.)
│   ├── llm_handler.py        model loading, async streaming, system prompt
│   └── session_store.py      SessionStore, ContextMemory, intent detection
├── Dockerfile                pre-built wheel, non-root user, HEALTHCHECK
├── docker-compose.yml        mounts ../models volume, sets env vars
├── requirements.txt
├── postman_collection.json   Postman v2.1 collection (REST + WebSocket)
└── api_service.ipynb         ← this notebook
```

### WebSocket message protocol

```
  CLIENT connects to  ws://localhost:8000/ws/chat
      ↓
  SERVER  → { type: session_created, session_id: ..., welcome: ... }
      ↓
  CLIENT  → { message: "I want to book a room" }       JSON
      ↓  (off-topic guard runs; if ok, LLM lock acquired)
  SERVER  → { type: token, content: "..." }            × N tokens (streaming)
  SERVER  → { type: done,  session_id: ..., intent: ..., latency_ms: ... }
      ↓  (repeat for more turns)
  CLIENT disconnects → session auto-deleted
```

---
## Cell 3 — Start the FastAPI Server

We launch `uvicorn` in a **daemon thread** so it runs alongside the notebook.
The server will be available at `http://localhost:8000`.

In [ ]:
import threading
import asyncio
import time
import sys
import os

# Ensure the Phase 4 directory is on sys.path so `api` package is importable
PHASE4_DIR = os.path.abspath(".")   # assumes notebook is in Phase 4/
if PHASE4_DIR not in sys.path:
    sys.path.insert(0, PHASE4_DIR)

import uvicorn

HOST = "127.0.0.1"
PORT = 8000

def _run_server():
    uvicorn.run(
        "api.main:app",
        host      = HOST,
        port      = PORT,
        log_level = "warning",   # keep notebook output clean
    )

server_thread = threading.Thread(target=_run_server, daemon=True)
server_thread.start()

# Wait for the server to be ready (model loading can take ~30 s on first run)
print("Starting server and loading model (this may take up to 60 s on first run) ...")
import httpx

for attempt in range(120):      # up to 120 × 1 s = 2 minutes
    time.sleep(1)
    try:
        r = httpx.get(f"http://{HOST}:{PORT}/health", timeout=2)
        data = r.json()
        if data.get("model_loaded"):
            print(f"\nServer ready after {attempt+1} s.")
            print(f"Status      : {data['status']}")
            print(f"Model       : {data['model_name']}")
            print(f"Sessions    : {data['active_sessions']}")
            break
        print(f"  [{attempt+1:3d}s] Model loading ...", end="\r")
    except Exception:
        print(f"  [{attempt+1:3d}s] Waiting for server ...", end="\r")
else:
    print("\nTimeout — check the console for errors.")

---
## Cell 4 — REST Tests: Health and Sessions

In [ ]:
import httpx
import json

BASE = f"http://{HOST}:{PORT}"


def pretty(r: httpx.Response) -> None:
    print(f"  {r.request.method} {r.request.url}  →  {r.status_code}")
    print(json.dumps(r.json(), indent=4))
    print()


print("=" * 55)
print("  REST API TESTS")
print("=" * 55)

# 1. Health
print("\n[1] GET /health")
pretty(httpx.get(f"{BASE}/health"))

# 2. Create session
print("[2] POST /sessions")
r = httpx.post(f"{BASE}/sessions")
pretty(r)
session_id = r.json()["session_id"]
print(f"   session_id saved: {session_id}")

# 3. List sessions
print("\n[3] GET /sessions")
pretty(httpx.get(f"{BASE}/sessions"))

# 4. Get session stats
print("\n[4] GET /sessions/{id}/stats")
pretty(httpx.get(f"{BASE}/sessions/{session_id}/stats"))

# 5. 404 on unknown session
print("\n[5] GET /sessions/fakeid/stats  (expect 404)")
r404 = httpx.get(f"{BASE}/sessions/fakeid/stats")
print(f"  Status: {r404.status_code}  {'PASS' if r404.status_code == 404 else 'FAIL'}")
print()

# 6. Delete session
print("\n[6] DELETE /sessions/{id}")
r_del = httpx.delete(f"{BASE}/sessions/{session_id}")
print(f"  Status: {r_del.status_code}  {'PASS' if r_del.status_code == 204 else 'FAIL'}")

---
## Cell 5 — WebSocket Test: Streaming Chat (Single Turn)

Opens a WebSocket connection, sends one message, and prints each token as it arrives.

In [ ]:
import asyncio
import json
import websockets

WS_URL = f"ws://{HOST}:{PORT}/ws/chat"


async def ws_single_turn(message: str):
    """Open a WS connection, send one message, collect all frames."""
    async with websockets.connect(WS_URL) as ws:
        # 1. Receive SessionCreated
        raw     = await ws.recv()
        welcome = json.loads(raw)
        print(f"  Session  : {welcome['session_id']}")
        print(f"  Welcome  : {welcome['welcome']}")
        print()

        # 2. Send guest message
        await ws.send(json.dumps({"message": message}))
        print(f"  Guest    : {message}")
        print(  "  Chiron   : ", end="", flush=True)

        # 3. Collect streamed tokens until done/error
        metadata = None
        while True:
            raw  = await ws.recv()
            msg  = json.loads(raw)
            mtype = msg.get("type")

            if mtype == "token":
                print(msg["content"], end="", flush=True)
            elif mtype == "done":
                metadata = msg
                break
            elif mtype == "error":
                print(f"\n  ERROR: [{msg['code']}] {msg['message']}")
                break

        print()
        if metadata:
            print(f"\n  Intent   : {metadata['intent']}")
            print(f"  Latency  : {metadata['latency_ms']:.0f} ms")
            print(f"  Memory   : {metadata['memory_stats']}")


print("=" * 55)
print("  WEBSOCKET TEST — Single Streaming Turn")
print("=" * 55)
asyncio.run(ws_single_turn("I'd like to know the pool hours and gym location."))

---
## Cell 6 — WebSocket Test: Off-Topic Guard

In [ ]:
print("=" * 55)
print("  WEBSOCKET TEST — Off-Topic Guard")
print("=" * 55)
print("(Off-topic messages should be redirected without calling the LLM)\n")

asyncio.run(ws_single_turn("What is the weather forecast for New York this week?"))

---
## Cell 7 — WebSocket Test: Multi-Turn Conversation

A 5-turn reservation dialogue over one persistent connection.

In [ ]:
MULTI_TURN = [
    "Hi, I'd like to book a room.",
    "Check-in August 5th, check-out August 8th.",
    "A double room for 2 guests.",
    "My name is Carlos Rivera, carlos@example.com",
    "Yes, please confirm the booking.",
]


async def ws_multi_turn(turns: list):
    async with websockets.connect(WS_URL, open_timeout=10) as ws:
        # Welcome frame
        w = json.loads(await ws.recv())
        print(f"  Session ID : {w['session_id']}\n")

        for i, msg in enumerate(turns, 1):
            await ws.send(json.dumps({"message": msg}))
            print(f"  [Turn {i}] Guest   : {msg}")
            print(         "  [Turn {i}] Chiron  : ".format(i=i), end="", flush=True)

            while True:
                raw  = await ws.recv()
                data = json.loads(raw)
                if data["type"] == "token":
                    print(data["content"], end="", flush=True)
                elif data["type"] in ("done", "error"):
                    print()
                    if data["type"] == "done":
                        print(f"           Intent  : {data['intent']}  |  "
                              f"Latency : {data['latency_ms']:.0f} ms")
                    else:
                        print(f"           ERROR: {data['message']}")
                    print()
                    break


print("=" * 60)
print("  WEBSOCKET TEST — 5-Turn Reservation Dialogue")
print("=" * 60)
asyncio.run(ws_multi_turn(MULTI_TURN))

---
## Cell 8 — WebSocket Test: Invalid JSON Error Handling

In [ ]:
async def ws_invalid_json_test():
    async with websockets.connect(WS_URL) as ws:
        await ws.recv()   # discard welcome frame

        # Send invalid JSON
        await ws.send("this is not valid json!!!")
        raw  = await ws.recv()
        resp = json.loads(raw)

        print(f"  Sent     : invalid JSON string")
        print(f"  Received : {resp}")
        passed = resp.get("type") == "error" and resp.get("code") == "invalid_request"
        print(f"  Result   : {'PASS' if passed else 'FAIL'}")

        # Connection should still be alive — send a valid message after
        await ws.send(json.dumps({"message": "Is the gym open 24 hours?"}))
        tokens = []
        while True:
            raw  = await ws.recv()
            data = json.loads(raw)
            if data["type"] == "token":
                tokens.append(data["content"])
            elif data["type"] in ("done", "error"):
                break
        reply = "".join(tokens)
        print(f"\n  Follow-up reply (connection survived): {reply[:80]}...")


print("=" * 55)
print("  WEBSOCKET TEST — Error Handling")
print("=" * 55)
asyncio.run(ws_invalid_json_test())

---
## Cell 9 — Concurrent Users Test

Spawns **3 simultaneous WebSocket connections** (3 virtual guests).
All requests queue behind the LLM lock — no crashes.

Expected behaviour:
- All 3 sessions receive a welcome frame immediately
- LLM inference is serialised (one at a time)
- All 3 get complete replies

In [ ]:
import asyncio

CONCURRENT_MSGS = [
    "What time does the spa open?",
    "Do you allow pets at the hotel?",
    "I want to cancel my reservation.",
]


async def single_ws_request(msg: str, label: str) -> dict:
    """One guest sends one message and returns the result."""
    import time as _time
    t0 = _time.perf_counter()

    async with websockets.connect(WS_URL, open_timeout=15) as ws:
        info = json.loads(await ws.recv())         # welcome
        sid  = info["session_id"]

        await ws.send(json.dumps({"message": msg}))
        tokens = []
        intent = "?"
        while True:
            raw  = await ws.recv()
            data = json.loads(raw)
            if data["type"] == "token":
                tokens.append(data["content"])
            elif data["type"] == "done":
                intent = data["intent"]
                break
            elif data["type"] == "error":
                return {"label": label, "sid": sid, "error": data["message"]}

    elapsed = (_time.perf_counter() - t0) * 1000
    return {
        "label"  : label,
        "sid"    : sid,
        "intent" : intent,
        "reply"  : "".join(tokens)[:80] + "...",
        "ms"     : round(elapsed),
    }


async def run_concurrent():
    tasks = [
        single_ws_request(msg, f"Guest-{i+1}")
        for i, msg in enumerate(CONCURRENT_MSGS)
    ]
    results = await asyncio.gather(*tasks)
    return results


print("=" * 60)
print("  CONCURRENT USERS TEST (3 simultaneous WebSocket clients)")
print("=" * 60)
print("Sending 3 requests at the same time ...\n")

results = asyncio.run(run_concurrent())

for r in results:
    if "error" in r:
        print(f"  [{r['label']}]  ERROR: {r['error']}")
    else:
        print(f"  [{r['label']}]  sid={r['sid']}  intent={r['intent']}  ms={r['ms']}")
        print(f"             {r['reply']}")
    print()

all_ok = all("error" not in r for r in results)
print(f"  Concurrent test: {'PASS — all {n} guests received replies.'.format(n=len(results)) if all_ok else 'FAIL'}")

---
## Cell 10 — Docker Deployment Instructions

Run these commands in a terminal from the `Phase 4/` directory.

### Build and start
```bash
# Build the image (only needed once or after code changes)
docker compose build

# Start the service (detached mode)
docker compose up -d

# Watch live logs
docker compose logs -f

# Stop the service
docker compose down
```

### Verify it's running
```bash
curl http://localhost:8000/health
```

### Environment variables (in `docker-compose.yml`)

| Variable | Default | Description |
|---|---|---|
| `MODEL_DIR` | `/models` | Path inside container where GGUF lives |
| `MAX_TOKENS` | `512` | Max generated tokens per reply |
| `TEMPERATURE` | `0.7` | Sampling temperature |
| `CONTEXT_SIZE` | `4096` | Model context window |
| `N_THREADS` | `4` | CPU cores for inference |
| `N_GPU_LAYERS` | `0` | GPU layers (0 = CPU only; 35 = full GPU) |

### Load the Postman collection
1. Open Postman → **Import** → choose `postman_collection.json`
2. Set the collection variable `base_url` to `http://localhost:8000`
3. Run the **REST** folder with the Collection Runner
4. For WebSocket tests, open each WS request in the **WebSocket Chat** folder

---
## Phase 4 Summary

| Component | Implementation |
|---|---|
| **Framework** | FastAPI 0.111+ with async/await |
| **WebSocket endpoint** | `ws://host:8000/ws/chat` — persistent, multi-turn |
| **Streaming** | `asyncio.Queue` bridges thread-pool LLM → async generator → WS |
| **Concurrency** | `asyncio.Lock` serialises LLM calls; multiple WS connections queue up |
| **Off-topic guard** | Keyword-based; redirects without calling LLM |
| **Session management** | In-memory `SessionStore`; REST CRUD + auto-cleanup on WS disconnect |
| **Error handling** | Malformed JSON, LLM errors, disconnects all handled gracefully |
| **Docker** | 2-stage build; pre-built wheel (no compiler); non-root user; HEALTHCHECK |
| **Postman** | 10 requests: REST × 5, WebSocket × 6, all with test scripts |